# Lumis 15: Does a Claude check make Lumis better?

Before a result is shown, Claude (vision) looks at the photo and answers: is skin visible, is the photo clear enough,
and which of the model's two most likely conditions the finding looks more like. The rule (fixed 2026-11-11, before
this notebook ran) lets Claude only make Lumis **more careful**: it can ask for a better photo, or lower the band
one step; it never raises a band or names a condition of its own. The code below is `backend/app/ml/claude_check.py`,
unchanged.

**Adoption rule (fixed in advance), on SCIN validation phone photos:** switch the check on only if
1. accuracy on the results Lumis shows (not referred, not rejected) rises by **at least 3 points**,
2. the share of photos that get a result falls by **no more than 15 points**, and
3. at least **95%** of API calls return a usable answer.

**Skin gate (second rule, fixed 2026-11-12 before this notebook ran, after notebook 10 showed the local photo check
catches only 35% of DermNet non-skin images and 4 of 20 of Zan's non-skin photos):** Claude's first answer alone
("is skin visible?") can be switched on by itself, as a "No skin detected" gate that never changes a band. Switch it on
only if
1. it catches **at least 80%** of non-skin images (DermNet's 170 removed non-skin images plus `photo_check/not_skin/`),
2. it wrongly rejects **at most 2%** of SCIN validation phone photos of the five conditions, and
3. at least **95%** of API calls return a usable answer.

`PART = 'validation'` decides both. `PART = 'test'` runs once on the SCIN test split, only after the check was adopted.

Needs a Colab secret named `ANTHROPIC_API_KEY` (key icon in the left sidebar; never paste the key into the notebook).
CPU runtime is fine. About 10 minutes for validation. Claude's answers are cached in Drive, so a rerun costs nothing.

In [ ]:
PART = 'validation'                 # 'validation' (decides) or 'test' (once, after adoption)
RUN = None                          # None = v2 if notebook 13 accepted it, otherwise v1
CLAUDE_MODEL = 'claude-haiku-5-5'
MIN_GAIN = 0.03
MAX_ANSWERED_DROP = 0.15
MIN_API_OK = 0.95
GATE_MIN_CAUGHT = 0.80              # skin gate: share of non-skin images it must catch
GATE_MAX_REJECTED = 0.02            # skin gate: share of real SCIN phone photos it may wrongly reject
NAMES = {'acne': 'Acne', 'eczema': 'Eczema', 'psoriasis': 'Psoriasis', 'scabies': 'Scabies',
         'tinea': 'Fungal skin infection (tinea or ringworm)'}          # same names as backend conditions.json

In [ ]:
!pip install -q anthropic simplejpeg
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'
API_KEY = userdata.get('ANTHROPIC_API_KEY')

In [ ]:
import os, json, zipfile, glob, time
import numpy as np, pandas as pd

OUT = f'{BASE}/exports/claude_check'
os.makedirs(OUT, exist_ok=True)
if PART == 'test':
    if os.path.exists(f'{OUT}/test_metrics.json'):
        raise SystemExit('The Claude check was already tested on the SCIN test split. It must only be used once.')
    decision = json.load(open(f'{OUT}/validation_decision.json'))
    if not (decision['enabled'] or decision.get('skin_gate')):
        raise SystemExit('Neither part of the check was adopted on validation, so there is nothing to test.')

if not os.path.exists('/content/scin'):
    with zipfile.ZipFile(f'{BASE}/data/scin_images.zip') as z:
        z.extractall('/content')

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    v2_runs = runs[runs.split == 'v2+scin']
    RUN = None
    if len(v2_runs):
        cand = v2_runs.iloc[-1]['run']
        p = f'{BASE}/exports/runs/{cand}/calibration.json'
        if os.path.exists(p) and json.load(open(p)).get('accepted'):
            RUN = cand
    if RUN is None:
        RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
cal = json.load(open(f'{BASE}/exports/runs/{RUN}/calibration.json'))
CLASSES = cal['classes']

scin = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
split_name = 'val' if PART == 'validation' else 'test'
photos = scin[(scin.split == split_name) & scin.eval_photo].reset_index(drop=True)
photos['file'] = '/content/' + photos.path
print(f'Model {RUN} | SCIN {split_name}: {len(photos)} cases', photos.lumis_class.value_counts().to_dict())

The next two cells are the backend code, unchanged.

In [ ]:
%%writefile predictor.py
"""Lumis skin-condition predictor.

Runs the exported EfficientNet-B0 (ONNX) and turns its output into what the app shows:
a condition, a calibrated confidence and a band (high / moderate / refer).

Preprocessing reproduces training exactly (ml/03_train.ipynb), checked pixel for pixel in ml/09_export.ipynb:
  1. Decode to RGB. JPEG uses libjpeg-turbo with the fast integer DCT, which is what
     tf.io.decode_image uses by default. PNG and WebP are decoded by Pillow.
  2. Resize to 224 x 224 like tf.image.resize: bilinear, half-pixel centers, no antialiasing,
     aspect ratio NOT kept (training stretched every image to a square).
  3. Cast to uint8 (truncation), then to float32 in the 0-255 range. EfficientNet scales and
     normalizes inside the model, so nothing else is done here.
One deliberate difference from training: phone photos are turned upright using their EXIF
orientation tag, because phones often save the pixels sideways.

Photo check (notebook 10), before any result is shown:
  no_skin           too few skin-colored pixels (the notebook 06 rule: YCbCr Cr 130-190, Cb 70-130)
  nothing_detected  the photo is unlike any of the training photos (energy score above the threshold)
Thresholds live in models/photo_check.json. Without that file every photo passes.

Output steps, decided in notebooks 04, 04b and 08:
  adjusted logits = logits + logit_adjust     (prior adjustment, training mix, tau 0.75)
  probabilities   = softmax(adjusted logits / temperature)
  band            = high if confidence >= high_from, moderate if >= refer_below, else refer
  scabies (any class in cap_at_moderate) is never shown as high: at most moderate.
"""
from __future__ import annotations

import hashlib
import io
import json
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import simplejpeg
from PIL import Image, UnidentifiedImageError

MAX_PIXELS = 50_000_000          # about 8,000 x 6,000; larger uploads are refused
SUPPORTED_FORMATS = {"JPEG", "PNG", "WEBP"}


class UnsupportedImageError(ValueError):
    """The upload is not an image Lumis can read."""


# ---------------------------------------------------------------- preprocessing

def _apply_exif_orientation(rgb: np.ndarray, orientation: int) -> np.ndarray:
    """Same result as PIL.ImageOps.exif_transpose, on a numpy array."""
    if orientation == 2:
        return rgb[:, ::-1]
    if orientation == 3:
        return rgb[::-1, ::-1]
    if orientation == 4:
        return rgb[::-1]
    if orientation == 5:
        return rgb.transpose(1, 0, 2)
    if orientation == 6:
        return np.rot90(rgb, -1)
    if orientation == 7:
        return rgb.transpose(1, 0, 2)[::-1, ::-1]
    if orientation == 8:
        return np.rot90(rgb, 1)
    return rgb


def decode_image(data: bytes, apply_exif: bool = True) -> np.ndarray:
    """Bytes of a JPEG, PNG or WebP file -> uint8 array of shape (height, width, 3)."""
    try:
        im = Image.open(io.BytesIO(data))
        fmt = im.format
        if fmt not in SUPPORTED_FORMATS:
            raise UnsupportedImageError(f"Unsupported image format: {fmt}. Use JPEG, PNG or WebP.")
        if im.width * im.height > MAX_PIXELS:
            raise UnsupportedImageError("Image is too large.")
        orientation = im.getexif().get(0x0112, 1) if apply_exif else 1
        rgb = None
        if fmt == "JPEG":
            try:
                rgb = simplejpeg.decode_jpeg(data, colorspace="RGB", fastdct=True, fastupsample=False)
            except ValueError:
                rgb = None       # unusual JPEGs (for example CMYK): fall back to Pillow below
        if rgb is None:
            rgb = np.asarray(im.convert("RGB"))
    except UnsupportedImageError:
        raise
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:
        raise UnsupportedImageError("The file could not be read as an image.") from exc
    return _apply_exif_orientation(rgb, orientation)


def resize_like_tf(img: np.ndarray, size: int) -> np.ndarray:
    """tf.image.resize(img, (size, size)) with its defaults: bilinear, half-pixel centers,
    no antialiasing. Returns float32."""
    h, w = img.shape[:2]

    def coords(n_in: int, n_out: int):
        src = (np.arange(n_out, dtype=np.float32) + 0.5) * np.float32(n_in / n_out) - 0.5
        lo = np.floor(src)
        frac = (src - lo).astype(np.float32)
        lo_i = np.clip(lo, 0, n_in - 1).astype(np.int64)
        hi_i = np.clip(lo + 1, 0, n_in - 1).astype(np.int64)
        return lo_i, hi_i, frac

    y0, y1, fy = coords(h, size)
    x0, x1, fx = coords(w, size)
    img = img.astype(np.float32)
    fx3, fy3 = fx[None, :, None], fy[:, None, None]
    top = img[y0][:, x0] * (1 - fx3) + img[y0][:, x1] * fx3
    bottom = img[y1][:, x0] * (1 - fx3) + img[y1][:, x1] * fx3
    return top * (1 - fy3) + bottom * fy3


def to_model_input(rgb: np.ndarray, size: int = 224) -> np.ndarray:
    """Decoded RGB array -> float32 model input of shape (1, size, size, 3), values 0-255."""
    resized = resize_like_tf(rgb, size).astype(np.uint8)       # truncation, as in training
    return resized.astype(np.float32)[None]


def preprocess(data: bytes, size: int = 224, apply_exif: bool = True) -> np.ndarray:
    """Image file bytes -> float32 model input of shape (1, size, size, 3), values 0-255."""
    return to_model_input(decode_image(data, apply_exif=apply_exif), size)


# ---------------------------------------------------------------- photo check

def skin_fraction(rgb: np.ndarray) -> float:
    """Share of skin-colored pixels, the rule from notebook 06: on a 128 x 128 copy, YCbCr chroma
    Cr 130-190 and Cb 70-130. Deliberately wide so red rashes and darker skin count as skin."""
    small = Image.fromarray(np.ascontiguousarray(rgb)).resize((128, 128)).convert("YCbCr")
    ycc = np.asarray(small, dtype=np.int16)
    cb, cr = ycc[..., 1], ycc[..., 2]
    return float(((cr >= 130) & (cr <= 190) & (cb >= 70) & (cb <= 130)).mean())


def energy_score(logits: np.ndarray) -> float:
    """Energy of the raw logits, -logsumexp (Liu et al., 2020). Photos like the training photos get low
    (more negative) energy; photos unlike anything the model was trained on get higher energy."""
    z = np.asarray(logits, dtype=np.float64).reshape(-1)
    m = z.max()
    return float(-(m + np.log(np.exp(z - m).sum())))


@dataclass(frozen=True)
class PhotoCheckConfig:
    min_skin_fraction: float
    max_energy: float

    @classmethod
    def load(cls, path: str | Path) -> "PhotoCheckConfig":
        c = json.loads(Path(path).read_text())
        return cls(min_skin_fraction=float(c["min_skin_fraction"]), max_energy=float(c["max_energy"]))

    def status(self, skin: float, energy: float) -> str:
        if skin < self.min_skin_fraction:
            return "no_skin"
        if energy > self.max_energy:
            return "nothing_detected"
        return "ok"


# ---------------------------------------------------------------- model output -> result

@dataclass(frozen=True)
class ModelConfig:
    model_version: str
    classes: list[str]
    img_size: int
    temperature: float
    logit_adjust: list[float]
    high_from: float
    refer_below: float
    cap_at_moderate: list[str] = field(default_factory=list)
    onnx_sha256: str | None = None
    model_file: str = "model.onnx"

    @classmethod
    def load(cls, path: str | Path) -> "ModelConfig":
        c = json.loads(Path(path).read_text())
        return cls(model_version=c["model_version"], classes=list(c["classes"]), img_size=int(c["img_size"]),
                   temperature=float(c["temperature"]), logit_adjust=[float(v) for v in c["logit_adjust"]],
                   high_from=float(c["bands"]["high_from"]), refer_below=float(c["bands"]["refer_below"]),
                   cap_at_moderate=list(c.get("cap_at_moderate", [])), onnx_sha256=c.get("onnx_sha256"),
                   model_file=c.get("model_file", "model.onnx"))


@dataclass(frozen=True)
class Prediction:
    condition: str              # most likely class (also given for 'refer', but the app shows it as inconclusive)
    confidence: float           # calibrated probability of that class, 0-1
    band: str                   # 'high' | 'moderate' | 'refer'
    probabilities: dict[str, float]
    capped: bool                # True if the scabies rule lowered the band from high to moderate
    model_version: str
    status: str = "ok"          # 'ok' | 'no_skin' | 'nothing_detected' (photo check)
    skin_fraction: float | None = None
    energy: float | None = None

    def to_dict(self) -> dict:
        return asdict(self)


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def postprocess(logits: np.ndarray, cfg: ModelConfig) -> Prediction:
    """Raw model logits for one image (shape (K,)) -> Prediction."""
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    probs = softmax((logits + np.asarray(cfg.logit_adjust)) / cfg.temperature)
    i = int(probs.argmax())
    conf = float(probs[i])
    condition = cfg.classes[i]
    if conf >= cfg.high_from:
        band = "high"
    elif conf >= cfg.refer_below:
        band = "moderate"
    else:
        band = "refer"
    capped = band == "high" and condition in cfg.cap_at_moderate
    if capped:
        band = "moderate"
    return Prediction(condition=condition, confidence=round(conf, 4), band=band,
                      probabilities={c: round(float(p), 4) for c, p in zip(cfg.classes, probs)},
                      capped=capped, model_version=cfg.model_version)


# ---------------------------------------------------------------- the predictor

def sha256_of(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class LumisPredictor:
    """Load once at startup, then call predict() per upload. Safe to share across requests."""

    def __init__(self, model_path: str | Path, config_path: str | Path, verify_checksum: bool = True,
                 photo_check_path: str | Path | None = None):
        import onnxruntime as ort

        self.config = ModelConfig.load(config_path)
        self.photo_check = PhotoCheckConfig.load(photo_check_path) if photo_check_path else None
        if verify_checksum and self.config.onnx_sha256:
            actual = sha256_of(model_path)
            if actual != self.config.onnx_sha256:
                raise RuntimeError(f"Model file does not match its config (sha256 {actual[:12]}..., "
                                   f"expected {self.config.onnx_sha256[:12]}...). Re-download both from Drive.")
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    @classmethod
    def from_dir(cls, models_dir: str | Path, config_name: str = "lumis_model_config.json") -> "LumisPredictor":
        """Load the config in models_dir and the ONNX file it names."""
        models_dir = Path(models_dir)
        config_path = models_dir / config_name
        photo_check = models_dir / "photo_check.json"
        return cls(models_dir / ModelConfig.load(config_path).model_file, config_path,
                   photo_check_path=photo_check if photo_check.exists() else None)

    def logits(self, batch: np.ndarray) -> np.ndarray:
        return self.session.run(None, {self.input_name: batch.astype(np.float32)})[0]

    def predict(self, data: bytes) -> Prediction:
        rgb = decode_image(data)
        skin = skin_fraction(rgb)
        logits = self.logits(to_model_input(rgb, self.config.img_size))[0]
        energy = energy_score(logits)
        status = self.photo_check.status(skin, energy) if self.photo_check else "ok"
        return replace(postprocess(logits, self.config), status=status,
                       skin_fraction=round(skin, 4), energy=round(energy, 4))

In [ ]:
%%writefile claude_check.py
"""Claude check: a second look at each photo before Lumis shows a result.

Claude (vision) answers three structured questions about the photo:
  1. is skin clearly visible?                         no  -> "No skin detected."
  2. is the photo good enough to judge?               no  -> "Photo not clear enough." (retake)
  3. of the model's two most likely conditions, which one does the visible finding look more like
     (or neither, or cannot tell)?

Rule (decided 2026-11-11, before any evaluation): Claude can only make Lumis MORE careful.
  - Claude picks the model's top condition                -> result unchanged
  - Claude picks the second, "neither" or "cannot tell"   -> band lowered one step (high -> moderate -> refer)
  - Claude never raises a band and never introduces a condition of its own.
  - If the API fails, refuses, or times out, the check is skipped and the model's result stands (recorded).
Claude's answer is never shown to the user. The two candidates are given in alphabetical order, so their
order does not hint which one the model preferred.

Two parts, each switched on separately by notebook 15 on its own pre-declared rule (models/claude_check.json):
  "skin_gate": true  -> only question 1 is used (no skin -> "No skin detected"); bands are never changed.
  "enabled": true    -> all three questions are used, as described above.
"""
from __future__ import annotations

import base64
import io
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from PIL import Image

DEFAULT_MODEL = "claude-haiku-5-5"
MAX_EDGE = 1024              # about 1,300 image tokens; plenty to judge a skin photo

SYSTEM = ("You check photos for Lumis, a skin-condition screening aid. You do not diagnose, and your answer is "
          "never shown to the person who took the photo. Answer only with the requested JSON.")

PROMPT = """Look at this photo.
1. skin_visible: is human skin clearly visible and the main subject of the photo?
2. photo_quality: "good" if the skin area is in focus, close enough and lit well enough to judge its appearance; otherwise "poor", and give the main photo_problem.
3. A screening model says the photo most likely shows one of these two conditions:
   A) {a}
   B) {b}
   Which one is the visible skin finding more consistent with? Answer "A", "B", "neither" (it looks like something else, or like normal skin), or "cannot_tell"."""

SCHEMA = {
    "type": "object",
    "properties": {
        "skin_visible": {"type": "boolean"},
        "photo_quality": {"type": "string", "enum": ["good", "poor"]},
        "photo_problem": {"type": "string",
                          "enum": ["none", "blurry", "too_dark", "too_far", "obstructed", "not_skin", "other"]},
        "more_consistent_with": {"type": "string", "enum": ["A", "B", "neither", "cannot_tell"]},
    },
    "required": ["skin_visible", "photo_quality", "photo_problem", "more_consistent_with"],
    "additionalProperties": False,
}


@dataclass(frozen=True)
class ClaudeVerdict:
    ok: bool                         # False = check skipped (API error, refusal, bad output)
    skin_visible: bool | None = None
    photo_quality: str | None = None
    photo_problem: str | None = None
    choice: str | None = None        # a condition id, "neither" or "cannot_tell"
    error: str | None = None
    model: str | None = None

    def to_dict(self) -> dict:
        return asdict(self)


def prepare_image(rgb: np.ndarray, max_edge: int = MAX_EDGE) -> bytes:
    """Decoded RGB array (already turned upright) -> JPEG bytes, long edge at most max_edge."""
    im = Image.fromarray(np.ascontiguousarray(rgb))
    if max(im.size) > max_edge:
        im.thumbnail((max_edge, max_edge), Image.LANCZOS)
    b = io.BytesIO()
    im.save(b, "JPEG", quality=90)
    return b.getvalue()


def build_request(jpeg: bytes, candidates: list[str], names: dict[str, str], model: str) -> dict:
    a, b = sorted(candidates)
    return {
        "model": model,
        "max_tokens": 200,
        "system": SYSTEM,
        "messages": [{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                         "data": base64.standard_b64encode(jpeg).decode()}},
            {"type": "text", "text": PROMPT.format(a=names.get(a, a), b=names.get(b, b))},
        ]}],
        "extra_body": {"output_config": {"format": {"type": "json_schema", "schema": SCHEMA}}},
    }


def parse_response(response, candidates: list[str], model: str) -> ClaudeVerdict:
    if getattr(response, "stop_reason", None) == "refusal":
        return ClaudeVerdict(ok=False, error="refusal", model=model)
    text = next((blk.text for blk in response.content if getattr(blk, "type", None) == "text"), None)
    try:
        d = json.loads(text)
        pick = str(d["more_consistent_with"]).strip()
        a, b = sorted(candidates)
        choice = {"a": a, "b": b, "neither": "neither", "cannot_tell": "cannot_tell"}[pick.lower()]
        return ClaudeVerdict(ok=True, skin_visible=bool(d["skin_visible"]),
                             photo_quality=str(d["photo_quality"]).lower(),
                             photo_problem=str(d["photo_problem"]).lower(), choice=choice, model=model)
    except (TypeError, ValueError, KeyError) as exc:
        return ClaudeVerdict(ok=False, error=f"unreadable output: {type(exc).__name__}", model=model)


class ClaudeChecker:
    def __init__(self, api_key: str, model: str = DEFAULT_MODEL, timeout: float = 20.0, client=None,
                 skin_gate: bool = True, full: bool = True):
        if client is None:
            import anthropic
            client = anthropic.Anthropic(api_key=api_key, timeout=timeout, max_retries=1)
        self.client, self.model = client, model
        self.skin_gate, self.full = skin_gate, full          # which parts of the rule are switched on

    def check(self, rgb: np.ndarray, candidates: list[str], names: dict[str, str]) -> ClaudeVerdict:
        try:
            response = self.client.messages.create(**build_request(prepare_image(rgb), candidates, names, self.model))
        except Exception as exc:                       # network, rate limit, server error: skip the check
            return ClaudeVerdict(ok=False, error=f"api error: {type(exc).__name__}", model=self.model)
        return parse_response(response, candidates, self.model)


LOWER = {"high": "moderate", "moderate": "refer", "refer": "refer"}


def apply_verdict(band: str, top: str, verdict: ClaudeVerdict, skin_gate: bool = True,
                  full: bool = True) -> tuple[str, str]:
    """(model band, model top condition, verdict) -> (status, band). Status is 'ok', 'no_skin' or 'poor_photo'.
    Never raises a band. skin_gate / full say which parts of the rule are switched on."""
    if not verdict.ok:
        return "ok", band
    if (skin_gate or full) and verdict.skin_visible is False:
        return "no_skin", band
    if not full:
        return "ok", band
    if verdict.photo_quality == "poor":
        return "poor_photo", band
    if verdict.choice == top:
        return "ok", band
    return "ok", LOWER[band]


@dataclass(frozen=True)
class ClaudeCheckConfig:
    enabled: bool                    # full check (notebook 15, section 2)
    model: str = DEFAULT_MODEL
    skin_gate: bool = False          # "no skin" question only (notebook 15, section 3)

    @property
    def any(self) -> bool:
        return self.enabled or self.skin_gate

    @classmethod
    def load(cls, path: str | Path) -> "ClaudeCheckConfig":
        p = Path(path)
        if not p.exists():
            return cls(enabled=False)
        c = json.loads(p.read_text())
        return cls(enabled=bool(c.get("enabled", False)), model=c.get("model", DEFAULT_MODEL),
                   skin_gate=bool(c.get("skin_gate", False)))

In [ ]:
import importlib, predictor, claude_check
importlib.reload(predictor); importlib.reload(claude_check)
from predictor import decode_image
from claude_check import ClaudeChecker, ClaudeVerdict, apply_verdict
import tensorflow as tf, keras

model = keras.models.load_model(info.model_file)

def readable(files):
    ok = []
    for f in files:
        try:
            tf.io.decode_image(tf.io.read_file(f), channels=3, expand_animations=False)
            ok.append(f)
        except Exception:
            print('Skipped, cannot be read (HEIC must be saved as JPEG):', os.path.basename(f))
    return ok

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.cast(tf.cast(tf.image.resize(img, (224, 224)), tf.uint8), tf.float32)
    return keras.applications.efficientnet.preprocess_input(img)

def model_results(files):
    ds = tf.data.Dataset.from_tensor_slices(files).map(load).batch(32)
    lg = model.predict(ds, verbose=0)
    z = (lg + np.array(cal['logit_adjust'])) / cal['temperature']
    probs = np.exp(z - z.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
    pred, conf = probs.argmax(1), probs.max(1)
    b = cal['bands']
    band = np.where(conf >= b['high_from'], 'high', np.where(conf >= b['refer_below'], 'moderate', 'refer'))
    band = np.where((pred == CLASSES.index('scabies')) & (band == 'high'), 'moderate', band)
    top2 = [[CLASSES[i] for i in np.argsort(-p)[:2]] for p in probs]
    return pred, band, top2

checker = ClaudeChecker(API_KEY, CLAUDE_MODEL)
CACHE = f'{OUT}/answers_{CLAUDE_MODEL}_{RUN}.jsonl'
cache = {}
if os.path.exists(CACHE):
    for line in open(CACHE):
        r = json.loads(line)
        cache[(r['file'], tuple(r['top2']))] = r['verdict']

def claude_for(files, top2s):
    out = []
    with open(CACHE, 'a') as f:
        for file, top2 in zip(files, top2s):
            key = (file, tuple(top2))
            if key not in cache:
                v = checker.check(decode_image(open(file, 'rb').read()), top2, NAMES)
                cache[key] = v.to_dict()
                f.write(json.dumps({'file': file, 'top2': top2, 'verdict': cache[key]}) + '\n')
                time.sleep(0.2)
            out.append(ClaudeVerdict(**cache[key]))
    return out

## 1. SCIN phone photos: model alone versus model + Claude check

In [ ]:
t0 = time.time()
pred, band, top2 = model_results(list(photos.file))
y = np.array([CLASSES.index(c) for c in photos.lumis_class])
verdicts = claude_for(list(photos.file), top2)
print(f'{len(verdicts)} checks in {time.time() - t0:.0f} s')

after = [apply_verdict(b, CLASSES[p], v) for b, p, v in zip(band, pred, verdicts)]
status2 = np.array([s for s, _ in after]); band2 = np.array([b for _, b in after])
correct = pred == y

def summary(shown, b):
    return {'share_shown': shown.mean(), 'accuracy_when_shown': correct[shown].mean() if shown.any() else np.nan,
            'high_share': (b == 'high').mean(),
            'high_accuracy': correct[b == 'high'].mean() if (b == 'high').any() else np.nan}

base = summary(band != 'refer', band)
with_c = summary((band2 != 'refer') & (status2 == 'ok'), np.where(status2 == 'ok', band2, 'rejected'))
table = pd.DataFrame({'model alone': base, 'model + Claude check': with_c}).round(3)
display(table)

api_ok = np.mean([v.ok for v in verdicts])
choice = pd.Series([('model top' if v.choice == CLASSES[p] else v.choice if v.choice in ('neither', 'cannot_tell') else 'model 2nd')
                    if v.ok else 'skipped' for v, p in zip(verdicts, pred)])
print('Claude answered:', f'{api_ok:.1%} of calls | photo outcome:', {k: int(v) for k, v in pd.Series(status2).value_counts().items()})
print('Claude choice:', choice.value_counts().to_dict())
print('Model accuracy when Claude picks the model top:', round(float(correct[choice.values == 'model top'].mean()), 3) if (choice == 'model top').any() else None,
      '| otherwise:', round(float(correct[choice.values != 'model top'].mean()), 3) if (choice != 'model top').any() else None)
errs = [v.error for v in verdicts if not v.ok]
if errs:
    print('Errors:', pd.Series(errs).value_counts().to_dict())

## 2. Full check: decision (validation) or result (test)

In [ ]:
gain = with_c['accuracy_when_shown'] - base['accuracy_when_shown']
drop = base['share_shown'] - with_c['share_shown']
checks = {'accuracy gain': (round(float(gain), 4), gain >= MIN_GAIN),
          'drop in share shown': (round(float(drop), 4), drop <= MAX_ANSWERED_DROP),
          'API calls answered': (round(float(api_ok), 4), api_ok >= MIN_API_OK)}
for k, (v, ok) in checks.items():
    print(f'{k:22s} {v:+.3f}  {"meets the rule" if ok else "FAILS the rule"}')
enabled = all(ok for _, ok in checks.values())
record = {'part': PART, 'run': RUN, 'claude_model': CLAUDE_MODEL, 'n': int(len(photos)),
          'model_alone': {k: float(v) for k, v in base.items()},
          'with_claude': {k: float(v) for k, v in with_c.items()},
          'api_ok': float(api_ok), 'rejected': {k: int(v) for k, v in pd.Series(status2).value_counts().items()},
          'choices': choice.value_counts().to_dict(), 'checks': {k: v[0] for k, v in checks.items()},
          'created_at': time.strftime('%Y-%m-%d %H:%M')}
print('\nFull check:', 'meets its rule' if enabled else 'does NOT meet its rule')

## 3. Skin gate: does Claude tell skin from non-skin?

Non-skin images: DermNet's removed non-skin images (notebook 06) and your `photo_check/not_skin/` photos.
A failed API call counts as not caught.

In [ ]:
scin_rejected = float(np.mean([v.ok and v.skin_visible is False for v in verdicts]))
if PART == 'validation':
    if not os.path.exists('/content/dermnet'):
        with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
            z.extractall('/content/dermnet')
    removed = pd.read_csv(f'{BASE}/exports/removed_nonskin_v2.csv')
    sources = {'DermNet non-skin': readable(['/content/dermnet/' + p for p in removed.path]),
               'Your not-skin photos': readable(sorted(f for f in glob.glob(f'{BASE}/data/photo_check/not_skin/*') if os.path.isfile(f)))}
    caught_by, gate_verdicts = {}, []
    for name, files in sources.items():
        if not files:
            continue
        _, _, t_ = model_results(files)
        vs = claude_for(files, t_)
        gate_verdicts += vs
        caught_by[name] = (sum(v.ok and v.skin_visible is False for v in vs), len(vs))
        print(f'{name}: caught {caught_by[name][0]} of {caught_by[name][1]}')
    caught = sum(c for c, _ in caught_by.values()) / max(sum(n for _, n in caught_by.values()), 1)
    gate_api_ok = float(np.mean([v.ok for v in gate_verdicts + verdicts]))
    gate_checks = {'non-skin images caught': (round(caught, 4), caught >= GATE_MIN_CAUGHT),
                   'SCIN photos wrongly rejected': (round(scin_rejected, 4), scin_rejected <= GATE_MAX_REJECTED),
                   'API calls answered': (round(gate_api_ok, 4), gate_api_ok >= MIN_API_OK)}
    for k, (v, ok) in gate_checks.items():
        print(f'{k:30s} {v:.3f}  {"meets the rule" if ok else "FAILS the rule"}')
    skin_gate = all(ok for _, ok in gate_checks.values())
    record['skin_gate_checks'] = {k: v[0] for k, v in gate_checks.items()}
    record['skin_gate_caught_by_source'] = {k: list(v) for k, v in caught_by.items()}
    print('\nSkin gate:', 'meets its rule' if skin_gate else 'does NOT meet its rule')
else:
    record['scin_test_wrongly_rejected_as_no_skin'] = scin_rejected
    print(f'SCIN test phone photos wrongly rejected as no skin: {scin_rejected:.1%}')

## 4. Save

In [ ]:
if PART == 'validation':
    record['enabled'], record['skin_gate'] = enabled, skin_gate
    json.dump(record, open(f'{OUT}/validation_decision.json', 'w'), indent=1, default=float)
    json.dump({'enabled': enabled, 'skin_gate': skin_gate, 'model': CLAUDE_MODEL, 'decided_on': 'SCIN validation',
               'run': RUN, 'rule': {'min_gain': MIN_GAIN, 'max_answered_drop': MAX_ANSWERED_DROP, 'min_api_ok': MIN_API_OK,
                                    'gate_min_caught': GATE_MIN_CAUGHT, 'gate_max_rejected': GATE_MAX_REJECTED}},
              open(f'{BASE}/exports/model/claude_check.json', 'w'), indent=1)
    print('Full check:', 'ADOPTED' if enabled else 'not adopted', '| Skin gate:', 'ADOPTED' if skin_gate else 'not adopted')
    print('Download exports/model/claude_check.json into backend/models/ (it says which parts are on).')
else:
    json.dump(record, open(f'{OUT}/test_metrics.json', 'w'), indent=1, default=float)
    print('Saved the one-time test result.')

## 5. Your own photos (validation only): does Claude spot healthy skin and non-skin?

Reported only; not part of the decision.

In [ ]:
if PART == 'validation':
    rows = []
    for sub in ['healthy_skin', 'not_skin', 'real_conditions']:
        files = readable(sorted(f for f in glob.glob(f'{BASE}/data/photo_check/{sub}/*') if os.path.isfile(f)))
        if not files:
            continue
        p_, b_, t_ = model_results(files)
        for f, p, b, v in zip(files, p_, b_, claude_for(files, t_)):
            s, b2 = apply_verdict(b, CLASSES[p], v)
            rows.append({'set': sub, 'file': os.path.basename(f), 'model says': f'{CLASSES[p]} ({b})',
                         'claude skin': v.skin_visible, 'claude quality': v.photo_quality, 'claude choice': v.choice,
                         'final': s if s != 'ok' else b2})
    own = pd.DataFrame(rows)
    if len(own):
        display(own.groupby('set').final.value_counts().unstack(fill_value=0))
        own.to_csv(f'{OUT}/own_photos.csv', index=False)